# Now testing with England/UK

Have worked out how to plot H3 population density with pydeck in `pydeck_test.ipynb` and have arrived at a few different funcitones, shown below

In [120]:
import pydeck as pdk
import pandas as pd
import numpy as np
import h3
import io
import requests
import geodatasets as gds
import geopandas as gpd
import numpy as np
import pycountry
from geopy.geocoders import Nominatim
from pathlib import Path
import gzip, shutil

In [4]:
def pop_density_calc(df):
    # First should probs check that it is the right dataframe input
    if df.columns.to_list()[0:2] != ["h3", "population"]:
        raise ValueError("Nu uh uh, wrong dataframe initial two columns name, should be ['h3', 'population', ...]")
    
    # To be honest, probs wont trip the above, but only problems will be if there's an alpha channel already or a population density already produced
    # Should probs add more error tripping
    
    area_km_calc_format = lambda x: h3.cell_area(x,"km^2")
    df["population/km^2"] = df["population"] / (df["h3"].map(area_km_calc_format))
    return df

In [5]:
def to_parent_aggregate_3(df, required_res):
    if required_res >= 8:
        raise ValueError("why are u doing that. The maximum resolution data we have is 8 so this is silly. Choose < 8")
    df_temp = df.copy(deep=True)

    # Check if all h3 res 8 cells
    get_res = lambda x: h3.get_resolution(x)
    res = pd.Series(data=df_temp["h3"].map(get_res),name="res")
    
    if not res.eq(8).all():
        raise ValueError(f"Wrong resolution for one/some input H3 hexes, supposed to be res. 8")
    
    to_parent_format = lambda x: h3.cell_to_parent(x, required_res)
    df_temp["h3"] = df["h3"].map(to_parent_format)
    df_temp = df_temp.groupby("h3")["population"].sum().reset_index()
    df_temp = pop_density_calc(df_temp)
    return df_temp

In [6]:
def get_country_coords(country_name):
    geolocator = Nominatim(user_agent="GeospatialCancerAccess2")
    try:
        location = geolocator.geocode(country_name)
        if location:
            return (location.latitude, location.longitude)
        else:
            return None
    except Exception as e:
        print(f"Error: {e}")
        return None

In [7]:
def plot_pop_density_log_opacity(
        df, 
        country,
        min_pop_dens=1, 
        max_pop_dens=50000,
        h3_column="h3",
        pickable=True,
        stroked=False,
        filled=True,
        extruded=False,
        get_elevation="population/km^2",
        elevation_scale=20,
        high_precision=True,
        auto_highlight=True,
        pitch=30,
        bearing=0,
        map_provider="carto",
        map_style="light", # road, dark, satellite, dark_no_labels, light_no_labels,
        map_output_folder="H3_pydeck_maps",
        map_name="log_opacity_map",
        open_browser=True,
        ):

    lp = np.log(df["population/km^2"].clip(lower=min_pop_dens, upper=max_pop_dens))
    lp_min, lp_max = np.log(min_pop_dens), np.log(max_pop_dens)

    z = (lp - lp_min) / (lp_max - lp_min) # in [0, 1]

    df_temp= df.copy(deep=True)

    df_temp["alpha"] = (30 + z * (255 - 30)).round().astype(int)
    # Purely for formatting and displaying in the tooltip
    df_temp["population_dens_2dp"] = df_temp["population/km^2"].map(lambda x: round(x,2))

    layer = pdk.Layer(
        "H3HexagonLayer",
        df_temp,
        pickable=pickable,
        stroked=stroked,
        filled=filled,
        extruded=extruded,
        get_elevation=get_elevation,
        elevation_scale=elevation_scale,
        high_precision=high_precision,
        get_hexagon=h3_column,
        auto_highlight=auto_highlight,
        get_fill_color="[0, 122, 255, alpha]",  # data-driven transparency
        get_line_color=[255, 255, 255],
        line_width_min_pixels=1,
    )

    country_name = pycountry.countries.lookup(country).name
    latitude, longitude = get_country_coords(country_name)

    view_state = pdk.ViewState(latitude=latitude, longitude=longitude, zoom=7, bearing=bearing, pitch=pitch)

    tooltip = {
        "html": "Population: {population},<br/>Population density: {population_dens_2dp}/km^2",
        "style": {
            "backgroundColor": "steelblue",
            "color": "white"
        }
    }


    r = pdk.Deck(layers=[layer], 
                initial_view_state=view_state,
                map_provider=map_provider,
                map_style=map_style,  # road, dark, satellite, dark_no_labels, light_no_labels
                tooltip=tooltip)

    Path(map_output_folder).mkdir(parents=True, exist_ok=True)
    
    filename = Path(map_output_folder) / Path(map_name + '.html')

    r.to_html(filename=filename, open_browser=open_browser, notebook_display=False)

End of functions arrived at from `pydeck_test.ipynb`

Now we want to download the England/UK pop density `.gpkg` file, load it into GeoPandas, format it correctly, and see if first we can plot the population density as we have done before.

I will note that 
* I am currently not super sure whether keeping the data as a GeoDataFrame in GeoPandas (which keeps a geometry column we can do operations on) will be necessary or whether we can use a DataFrame and then just use H3 funcitons on this dataframe
* I think it's probably best to keep in the long run so we can do alternative plotting to pydeck as well as these use the geometry column.
* if pydeck supports using a GeoDataFrame then will do this, just need to check.

Downloading and extracting UK

In [8]:
def unzip_gpkg(gz_path):
    gpkg_path = gz_path.with_suffix("")

    # Unzip
    with gzip.open(gz_path, 'rb') as f_in, open(gpkg_path, "wb") as f_out:
        shutil.copyfileobj(f_in, f_out)
    return gpkg_path

def download_H3_population_density_gpkg(
        country_name, 
        output_dir="H3_pop_density_maps", 
        progress_callback= None, 
        overwrite_download = False):
    
    Path(output_dir).mkdir(parents=True, exist_ok=True)

    try:
        selected_country = pycountry.countries.get(name = country_name)
        selected_country_alpha_2 = selected_country.alpha_2
    except LookupError:
        return False, f"Country, {country_name}, not found"
    
    base_url = (
    "https://geodata-eu-central-1-kontur-public.s3.amazonaws.com/kontur_datasets/"
    "kontur_population_{country_alpha_2}_20231101.gpkg.gz"
    )

    target_url = base_url.format(country_alpha_2=selected_country_alpha_2)
    # Path of downloaded zipped file
    gz_path = Path(output_dir)/ f"{selected_country_alpha_2}_H3_population_density_map.gpkg.gz"
    # Path of final unzipped file
    output_file = gz_path.with_suffix("")


    # Check if file exists and we shouldn't overwrite
    if not overwrite_download and Path(output_file).exists():
        return True, output_file, f"File already exists at:\n{output_file}"

    try:
        # Use session to enable connection pooling and compression
        with requests.session() as session:

            with session.get(target_url, stream=True) as response:
                response.raise_for_status()
                total_size = int(response.headers.get("Content-Length",0))

                with open(gz_path, mode="wb") as file:
                    
                    downloaded = 0
                    for chunk in response.iter_content(chunk_size= 10 * 1024):
                        file.write(chunk)
                        downloaded += len(chunk)
                        if progress_callback and total_size>0:
                            progress = int(100* downloaded/total_size)
                            progress_callback(progress)
    
    except requests.exceptions.RequestException as e:
        # Handle incomplete shit
        if Path(gz_path).exists() :
            Path(gz_path).unlink(missing_ok=True)
        return False, f"Download failed, {str(e)}"
    except Exception as e:
        return False, f"Error occured: {str(e)}"
    
    gpkg_path = unzip_gpkg(gz_path)
        
    # Delete .gz file
    Path(gz_path).unlink(missing_ok=True)

    return False, gpkg_path, f"File saved to {gpkg_path}"

In [9]:
# Test run
download_H3_population_density_gpkg("United Kingdom")

(True,
 PosixPath('H3_pop_density_maps/GB_H3_population_density_map.gpkg'),
 'File already exists at:\nH3_pop_density_maps/GB_H3_population_density_map.gpkg')

In [10]:
country_name="United Kingdom"
overwrite_download, gpkg_path, message = download_H3_population_density_gpkg(country_name)
print(overwrite_download,gpkg_path,message)
gdf = gpd.read_file(gpkg_path)

True H3_pop_density_maps/GB_H3_population_density_map.gpkg File already exists at:
H3_pop_density_maps/GB_H3_population_density_map.gpkg


In [11]:
gdf

,h3,population,geometry
0,881976db69fffff,31.0,"POLYGON ((-335766.921 7888754.141, -336617 788..."
1,881976db5dfffff,2.0,"POLYGON ((-330458.26 7886004.394, -331307.955 ..."
2,881976db4dfffff,22.0,"POLYGON ((-333216.709 7889458.693, -334066.776..."
3,881976db45fffff,27.0,"POLYGON ((-334194.819 7888345.291, -335044.819..."
4,881976db41fffff,9.0,"POLYGON ((-332622.995 7887936.163, -333472.916..."
...,...,...,...
224517,8809a09013fffff,6.0,"POLYGON ((-104771.747 8538965.139, -105672.291..."
224518,8809a0835bfffff,2.0,"POLYGON ((-87755.542 8532258.97, -88654.856 85..."
224519,8809a08269fffff,3.0,"POLYGON ((-91623.584 8534891.463, -92523.28 85..."
224520,8809a08267fffff,2.0,"POLYGON ((-88339.953 8533870.061, -89239.437 8..."


In [12]:
gdf = pop_density_calc(gdf)

In [14]:
plot_pop_density_log_opacity(gdf,country_name)

Error: HTTPSConnectionPool(host='nominatim.openstreetmap.org', port=443): Max retries exceeded with url: /search?q=United+Kingdom&format=json&limit=1 (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x306b355e0>, 'Connection to nominatim.openstreetmap.org timed out. (connect timeout=1)'))


TypeError: cannot unpack non-iterable NoneType object

Kinda takes long, lets see if is shorter with just the dataframe

In [31]:
df = pd.DataFrame(gdf.drop(columns="geometry"))
df

,h3,population,population/km^2
0,881976db69fffff,31.0,50.375906
1,881976db5dfffff,2.0,3.250767
2,881976db4dfffff,22.0,35.761391
3,881976db45fffff,27.0,43.880337
4,881976db41fffff,9.0,14.628296
...,...,...,...
224517,8809a09013fffff,6.0,10.622447
224518,8809a0835bfffff,2.0,3.544063
224519,8809a08269fffff,3.0,5.315573
224520,8809a08267fffff,2.0,3.544443


In [32]:
plot_pop_density_log_opacity(df,country_name)

Error: HTTPSConnectionPool(host='nominatim.openstreetmap.org', port=443): Max retries exceeded with url: /search?q=United+Kingdom&format=json&limit=1 (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x302f72540>, 'Connection to nominatim.openstreetmap.org timed out. (connect timeout=1)'))


TypeError: cannot unpack non-iterable NoneType object

Okay so 20 ish seconds shorter, just using dataframes not geodataframes

## GB Linac excel to dataframe testing (collapse and skip pls)

In [34]:
gb_linac = pd.read_excel("gbr_DIRAC.xlsx")
gb_linac

,City,Operator Name,He Photon And Electron Beam Rt,Proton Ion Therapy,XRay Generator,Brachy Therapy Inc El,Last Update,Coordinates
0,Country: United Kingdom (Count: 78),NaN,Total: 358,Total: 7,Total: 49,Total: 56,Latest: 2024,NaN
1,Aberdeen,NHS Grampian (Aberdeen Royal Infirmary),3,0,0,1,2018,"57.1551578,-2.1352878"
2,Basingstoke,University Hospitals Southampton NHS Foundatio...,1,0,0,0,2023,"51.2810411,-1.1108507"
3,Bath,Royal United Hospital Bath NHS Trust,2,0,1,0,2018,"51.3917, -2.3910"
4,Bedlington,"Rutherford Health Proton Beam Therapy, Northum...",0,1,0,0,2020,"55.002836, -1.591675"
...,...,...,...,...,...,...,...,...
74,Torquay,Torbay & South Devon Healthcare NHS Foundation...,2,0,0,0,2018,"50.4817461,-3.553979"
75,"Truro, Cornwall",Royal Cornwall Hospitals NHS Trust,2,0,1,0,2023,"50.2668819,-5.0931924"
76,Westcliff-on-Sea,Mid and South Essex NHS Foundation Trust (Sout...,4,0,0,1,2023,"51.5542562,0.6883494"
77,Wolverhampton,Royal Wolverhampton Hospitals NHS Trust,4,0,1,2,2016,"52.598801, -2.096532"


In [35]:
gb_linac_gdf = gpd.read_file("gbr_DIRAC.xlsx").drop(labels=0, axis=0)
gb_linac_gdf
# THIS JUST READS IN A DF AS THERE IS NO POLYGON GEOMETRY
# TF use this

gb_linac = pd.read_excel("gbr_DIRAC.xlsx").drop(labels=0, axis=0)
gb_linac

,City,Operator Name,He Photon And Electron Beam Rt,Proton Ion Therapy,XRay Generator,Brachy Therapy Inc El,Last Update,Coordinates
1,Aberdeen,NHS Grampian (Aberdeen Royal Infirmary),3,0,0,1,2018,"57.1551578,-2.1352878"
2,Basingstoke,University Hospitals Southampton NHS Foundatio...,1,0,0,0,2023,"51.2810411,-1.1108507"
3,Bath,Royal United Hospital Bath NHS Trust,2,0,1,0,2018,"51.3917, -2.3910"
4,Bedlington,"Rutherford Health Proton Beam Therapy, Northum...",0,1,0,0,2020,"55.002836, -1.591675"
5,Belfast,Belfast HSC Trust (Belfast City Hospital),10,0,0,1,2019,"54.587368,-5.9416654"
...,...,...,...,...,...,...,...,...
74,Torquay,Torbay & South Devon Healthcare NHS Foundation...,2,0,0,0,2018,"50.4817461,-3.553979"
75,"Truro, Cornwall",Royal Cornwall Hospitals NHS Trust,2,0,1,0,2023,"50.2668819,-5.0931924"
76,Westcliff-on-Sea,Mid and South Essex NHS Foundation Trust (Sout...,4,0,0,1,2023,"51.5542562,0.6883494"
77,Wolverhampton,Royal Wolverhampton Hospitals NHS Trust,4,0,1,2,2016,"52.598801, -2.096532"


In [36]:
gb_linac = pd.concat([gb_linac, gb_linac["Coordinates"].str.split(',', expand=True)], axis=1)

In [37]:
gb_linac

,City,Operator Name,He Photon And Electron Beam Rt,Proton Ion Therapy,XRay Generator,Brachy Therapy Inc El,Last Update,Coordinates,0,1
1,Aberdeen,NHS Grampian (Aberdeen Royal Infirmary),3,0,0,1,2018,"57.1551578,-2.1352878",57.1551578,-2.1352878
2,Basingstoke,University Hospitals Southampton NHS Foundatio...,1,0,0,0,2023,"51.2810411,-1.1108507",51.2810411,-1.1108507
3,Bath,Royal United Hospital Bath NHS Trust,2,0,1,0,2018,"51.3917, -2.3910",51.3917,-2.3910
4,Bedlington,"Rutherford Health Proton Beam Therapy, Northum...",0,1,0,0,2020,"55.002836, -1.591675",55.002836,-1.591675
5,Belfast,Belfast HSC Trust (Belfast City Hospital),10,0,0,1,2019,"54.587368,-5.9416654",54.587368,-5.9416654
...,...,...,...,...,...,...,...,...,...,...
74,Torquay,Torbay & South Devon Healthcare NHS Foundation...,2,0,0,0,2018,"50.4817461,-3.553979",50.4817461,-3.553979
75,"Truro, Cornwall",Royal Cornwall Hospitals NHS Trust,2,0,1,0,2023,"50.2668819,-5.0931924",50.2668819,-5.0931924
76,Westcliff-on-Sea,Mid and South Essex NHS Foundation Trust (Sout...,4,0,0,1,2023,"51.5542562,0.6883494",51.5542562,0.6883494
77,Wolverhampton,Royal Wolverhampton Hospitals NHS Trust,4,0,1,2,2016,"52.598801, -2.096532",52.598801,-2.096532


In [38]:
#gb_linac.pop("Coordinates")
gb_linac = gb_linac.rename(columns={"Coordinates":"coordinates"})

In [39]:
gb_linac = gb_linac.rename(columns={
    0: "latitude",
    1: "longitude",
})

In [40]:
gb_linac

,City,Operator Name,He Photon And Electron Beam Rt,Proton Ion Therapy,XRay Generator,Brachy Therapy Inc El,Last Update,coordinates,latitude,longitude
1,Aberdeen,NHS Grampian (Aberdeen Royal Infirmary),3,0,0,1,2018,"57.1551578,-2.1352878",57.1551578,-2.1352878
2,Basingstoke,University Hospitals Southampton NHS Foundatio...,1,0,0,0,2023,"51.2810411,-1.1108507",51.2810411,-1.1108507
3,Bath,Royal United Hospital Bath NHS Trust,2,0,1,0,2018,"51.3917, -2.3910",51.3917,-2.3910
4,Bedlington,"Rutherford Health Proton Beam Therapy, Northum...",0,1,0,0,2020,"55.002836, -1.591675",55.002836,-1.591675
5,Belfast,Belfast HSC Trust (Belfast City Hospital),10,0,0,1,2019,"54.587368,-5.9416654",54.587368,-5.9416654
...,...,...,...,...,...,...,...,...,...,...
74,Torquay,Torbay & South Devon Healthcare NHS Foundation...,2,0,0,0,2018,"50.4817461,-3.553979",50.4817461,-3.553979
75,"Truro, Cornwall",Royal Cornwall Hospitals NHS Trust,2,0,1,0,2023,"50.2668819,-5.0931924",50.2668819,-5.0931924
76,Westcliff-on-Sea,Mid and South Essex NHS Foundation Trust (Sout...,4,0,0,1,2023,"51.5542562,0.6883494",51.5542562,0.6883494
77,Wolverhampton,Royal Wolverhampton Hospitals NHS Trust,4,0,1,2,2016,"52.598801, -2.096532",52.598801,-2.096532


In [ ]:
gb_linac["latitude"] = gb_linac["latitude"].str.strip()
gb_linac["longitude"] = gb_linac["longitude"].str.strip()
gb_linac = gb_linac.rename(columns={"Operator Name": "name"})

In [42]:
gb_linac['coordinates'] = ('[' + gb_linac['coordinates']+']')
gb_linac


,City,name,He Photon And Electron Beam Rt,Proton Ion Therapy,XRay Generator,Brachy Therapy Inc El,Last Update,coordinates,latitude,longitude
1,Aberdeen,NHS Grampian (Aberdeen Royal Infirmary),3,0,0,1,2018,"[57.1551578,-2.1352878]",57.1551578,-2.1352878
2,Basingstoke,University Hospitals Southampton NHS Foundatio...,1,0,0,0,2023,"[51.2810411,-1.1108507]",51.2810411,-1.1108507
3,Bath,Royal United Hospital Bath NHS Trust,2,0,1,0,2018,"[51.3917, -2.3910]",51.3917,-2.3910
4,Bedlington,"Rutherford Health Proton Beam Therapy, Northum...",0,1,0,0,2020,"[55.002836, -1.591675]",55.002836,-1.591675
5,Belfast,Belfast HSC Trust (Belfast City Hospital),10,0,0,1,2019,"[54.587368,-5.9416654]",54.587368,-5.9416654
...,...,...,...,...,...,...,...,...,...,...
74,Torquay,Torbay & South Devon Healthcare NHS Foundation...,2,0,0,0,2018,"[50.4817461,-3.553979]",50.4817461,-3.553979
75,"Truro, Cornwall",Royal Cornwall Hospitals NHS Trust,2,0,1,0,2023,"[50.2668819,-5.0931924]",50.2668819,-5.0931924
76,Westcliff-on-Sea,Mid and South Essex NHS Foundation Trust (Sout...,4,0,0,1,2023,"[51.5542562,0.6883494]",51.5542562,0.6883494
77,Wolverhampton,Royal Wolverhampton Hospitals NHS Trust,4,0,1,2,2016,"[52.598801, -2.096532]",52.598801,-2.096532


In [43]:
#gb_linac_gdf.geometry.name
# Use to see error

In [44]:
del gb_linac["City"]

In [45]:
gb_linac

,name,He Photon And Electron Beam Rt,Proton Ion Therapy,XRay Generator,Brachy Therapy Inc El,Last Update,coordinates,latitude,longitude
1,NHS Grampian (Aberdeen Royal Infirmary),3,0,0,1,2018,"[57.1551578,-2.1352878]",57.1551578,-2.1352878
2,University Hospitals Southampton NHS Foundatio...,1,0,0,0,2023,"[51.2810411,-1.1108507]",51.2810411,-1.1108507
3,Royal United Hospital Bath NHS Trust,2,0,1,0,2018,"[51.3917, -2.3910]",51.3917,-2.3910
4,"Rutherford Health Proton Beam Therapy, Northum...",0,1,0,0,2020,"[55.002836, -1.591675]",55.002836,-1.591675
5,Belfast HSC Trust (Belfast City Hospital),10,0,0,1,2019,"[54.587368,-5.9416654]",54.587368,-5.9416654
...,...,...,...,...,...,...,...,...,...
74,Torbay & South Devon Healthcare NHS Foundation...,2,0,0,0,2018,"[50.4817461,-3.553979]",50.4817461,-3.553979
75,Royal Cornwall Hospitals NHS Trust,2,0,1,0,2023,"[50.2668819,-5.0931924]",50.2668819,-5.0931924
76,Mid and South Essex NHS Foundation Trust (Sout...,4,0,0,1,2023,"[51.5542562,0.6883494]",51.5542562,0.6883494
77,Royal Wolverhampton Hospitals NHS Trust,4,0,1,2,2016,"[52.598801, -2.096532]",52.598801,-2.096532


In [46]:
gb_linac["coordinates"] = gb_linac["coordinates"].str.replace(',', ', ')

In [47]:
gb_linac

,name,He Photon And Electron Beam Rt,Proton Ion Therapy,XRay Generator,Brachy Therapy Inc El,Last Update,coordinates,latitude,longitude
1,NHS Grampian (Aberdeen Royal Infirmary),3,0,0,1,2018,"[57.1551578, -2.1352878]",57.1551578,-2.1352878
2,University Hospitals Southampton NHS Foundatio...,1,0,0,0,2023,"[51.2810411, -1.1108507]",51.2810411,-1.1108507
3,Royal United Hospital Bath NHS Trust,2,0,1,0,2018,"[51.3917, -2.3910]",51.3917,-2.3910
4,"Rutherford Health Proton Beam Therapy, Northum...",0,1,0,0,2020,"[55.002836, -1.591675]",55.002836,-1.591675
5,Belfast HSC Trust (Belfast City Hospital),10,0,0,1,2019,"[54.587368, -5.9416654]",54.587368,-5.9416654
...,...,...,...,...,...,...,...,...,...
74,Torbay & South Devon Healthcare NHS Foundation...,2,0,0,0,2018,"[50.4817461, -3.553979]",50.4817461,-3.553979
75,Royal Cornwall Hospitals NHS Trust,2,0,1,0,2023,"[50.2668819, -5.0931924]",50.2668819,-5.0931924
76,Mid and South Essex NHS Foundation Trust (Sout...,4,0,0,1,2023,"[51.5542562, 0.6883494]",51.5542562,0.6883494
77,Royal Wolverhampton Hospitals NHS Trust,4,0,1,2,2016,"[52.598801, -2.096532]",52.598801,-2.096532


In [48]:
# need two rows for header index, mh for multi-header
#gb_linac_mh = pd.read_excel("gbr_DIRAC.xlsx", header=[0,1])
#gb_linac_mh

# Actually we just drop this second row in the above


In [49]:
from pydeck.types import String

MIN_POP, MAX_POP = 1, 50000.0  # pick what makes sense for your data
lp = np.log1p(df["population/km^2"].clip(lower=MIN_POP, upper=MAX_POP))
lp_min, lp_max = np.log1p(MIN_POP), np.log1p(MAX_POP)

z = (lp - lp_min) / (lp_max - lp_min)               # in [0, 1]

df_temp = df.copy(deep=True)

df_temp["alpha"] = (30 + z * (255 - 30)).round().astype(int)


h3 = pdk.Layer(
        "H3HexagonLayer",
        df_temp,
        pickable=True,
        stroked=False,
        filled=True,
        extruded=False,
        get_elevation="population/km^2",
        elevation_scale=20,
        high_precision=True,
        get_hexagon="h3",
        auto_highlight=True,
        get_fill_color="[0, 122, 255, alpha]",  # data-driven transparency
        get_line_color=[255, 255, 255],
        line_width_min_pixels=1,
    )

linacs = pdk.Layer(
        "TextLayer",
        data=gb_linac,
        get_position=['longitude','latitude'],
        get_text="name",
        get_size=12,
        get_color=[0, 0, 0],
        get_angle=0,
        get_text_anchor=String("start"),
        get_alignment_baseline=String("bottom"),
    )

view_state = pdk.ViewState(latitude=54.4, longitude=-3.5, zoom=7, bearing=0, pitch=30)

r = pdk.Deck(layers=[h3, linacs], 
             initial_view_state=view_state,
             map_provider="carto",
             map_style="light",  # road, dark, satellite, dark_no_labels, light_no_labels
             tooltip={"text": "Population: {population/km^2}"})

r.to_html(filename="H3_maps/log_opacity_england_test.html", open_browser=True, notebook_display=False)


In [50]:
linacs = pdk.Layer(
        "TextLayer",
        gb_linac,
        get_position="['longitude', 'latitude']",
        get_text="name",
        get_size=12,
        get_color=[0, 0, 0],
        get_angle=0,
        #get_text_anchor=String("start"),
        #get_alignment_baseline=String("bottom"),
    )

view_state = pdk.ViewState(latitude=54.4, longitude=-3.5, zoom=7, bearing=0, pitch=30)

r = pdk.Deck(layers=[linacs], 
             initial_view_state=view_state,
             map_provider="carto",
             map_style="light",  # road, dark, satellite, dark_no_labels, light_no_labels
)

r.to_html(filename="H3_maps/log_opacity_england_test.html", open_browser=True, notebook_display=False)


In [51]:
gb_linac

,name,He Photon And Electron Beam Rt,Proton Ion Therapy,XRay Generator,Brachy Therapy Inc El,Last Update,coordinates,latitude,longitude
1,NHS Grampian (Aberdeen Royal Infirmary),3,0,0,1,2018,"[57.1551578, -2.1352878]",57.1551578,-2.1352878
2,University Hospitals Southampton NHS Foundatio...,1,0,0,0,2023,"[51.2810411, -1.1108507]",51.2810411,-1.1108507
3,Royal United Hospital Bath NHS Trust,2,0,1,0,2018,"[51.3917, -2.3910]",51.3917,-2.3910
4,"Rutherford Health Proton Beam Therapy, Northum...",0,1,0,0,2020,"[55.002836, -1.591675]",55.002836,-1.591675
5,Belfast HSC Trust (Belfast City Hospital),10,0,0,1,2019,"[54.587368, -5.9416654]",54.587368,-5.9416654
...,...,...,...,...,...,...,...,...,...
74,Torbay & South Devon Healthcare NHS Foundation...,2,0,0,0,2018,"[50.4817461, -3.553979]",50.4817461,-3.553979
75,Royal Cornwall Hospitals NHS Trust,2,0,1,0,2023,"[50.2668819, -5.0931924]",50.2668819,-5.0931924
76,Mid and South Essex NHS Foundation Trust (Sout...,4,0,0,1,2023,"[51.5542562, 0.6883494]",51.5542562,0.6883494
77,Royal Wolverhampton Hospitals NHS Trust,4,0,1,2,2016,"[52.598801, -2.096532]",52.598801,-2.096532


In [52]:
gb_linac["coordinates"] = ('[' + gb_linac['longitude']+', ' + gb_linac['latitude'] + ']')

In [53]:
gb_linac

,name,He Photon And Electron Beam Rt,Proton Ion Therapy,XRay Generator,Brachy Therapy Inc El,Last Update,coordinates,latitude,longitude
1,NHS Grampian (Aberdeen Royal Infirmary),3,0,0,1,2018,"[-2.1352878, 57.1551578]",57.1551578,-2.1352878
2,University Hospitals Southampton NHS Foundatio...,1,0,0,0,2023,"[-1.1108507, 51.2810411]",51.2810411,-1.1108507
3,Royal United Hospital Bath NHS Trust,2,0,1,0,2018,"[-2.3910, 51.3917]",51.3917,-2.3910
4,"Rutherford Health Proton Beam Therapy, Northum...",0,1,0,0,2020,"[-1.591675, 55.002836]",55.002836,-1.591675
5,Belfast HSC Trust (Belfast City Hospital),10,0,0,1,2019,"[-5.9416654, 54.587368]",54.587368,-5.9416654
...,...,...,...,...,...,...,...,...,...
74,Torbay & South Devon Healthcare NHS Foundation...,2,0,0,0,2018,"[-3.553979, 50.4817461]",50.4817461,-3.553979
75,Royal Cornwall Hospitals NHS Trust,2,0,1,0,2023,"[-5.0931924, 50.2668819]",50.2668819,-5.0931924
76,Mid and South Essex NHS Foundation Trust (Sout...,4,0,0,1,2023,"[0.6883494, 51.5542562]",51.5542562,0.6883494
77,Royal Wolverhampton Hospitals NHS Trust,4,0,1,2,2016,"[-2.096532, 52.598801]",52.598801,-2.096532


In [54]:
fuck_off = gb_linac.copy(deep=True)

In [55]:
del fuck_off[]

SyntaxError: invalid syntax (1355082458.py, line 1)

In [ ]:

TEXT_LAYER_DATA = "https://raw.githubusercontent.com/visgl/deck.gl-data/master/website/bart-stations.json"  # noqa
df_test = pd.read_json(TEXT_LAYER_DATA)

In [ ]:
"""
TextLayer
========

Names of various public transit stops within San Francisco, plotted at the location of that stop
"""

import pydeck as pdk
from pydeck.types import String
import pandas as pd

TEXT_LAYER_DATA = "https://raw.githubusercontent.com/visgl/deck.gl-data/master/website/bart-stations.json"  # noqa
df = pd.read_json(TEXT_LAYER_DATA)

# Define a layer to display on a map
layer = pdk.Layer(
    "TextLayer",
    df,
    pickable=True,
    get_position='coordinates',
    get_text='name',
    get_size=16,
    get_color=[0, 0, 0],
    get_angle=0,
    # Note that string constants in pydeck are explicitly passed as strings
    # This distinguishes them from columns in a data set
    get_text_anchor=String("middle"),
    get_alignment_baseline=String("center"),
)

# Set the viewport location
view_state = pdk.ViewState(latitude=37.7749295, longitude=-122.4194155, zoom=10, bearing=0, pitch=45)

# Render
r = pdk.Deck(
    layers=[layer],
    initial_view_state=view_state,
    tooltip={"text": "{name}\n{address}"},
    map_style=pdk.map_styles.ROAD,
)
r.to_html("text_layer.html", open_browser=True, notebook_display=False)

In [ ]:
df["coordinates"]

0     [-122.123801, 37.893394]
1     [-122.271604, 37.803664]
2     [-122.419694, 37.765062]
3      [-122.269029, 37.80787]
4     [-122.418466, 37.752254]
5      [-122.26978, 37.853024]
6     [-122.447414, 37.721981]
7     [-122.126871, 37.697185]
8     [-122.075567, 37.690754]
9     [-122.413756, 37.779528]
10    [-122.466233, 37.684638]
11    [-122.197273, 37.754006]
12    [-122.029095, 37.973737]
13    [-122.469081, 37.706121]
14    [-122.268045, 37.869867]
15    [-122.317269, 37.925655]
16    [-121.900367, 37.701695]
17    [-122.396742, 37.792976]
18      [-121.9764, 37.557355]
19    [-122.224274, 37.774963]
20    [-122.434092, 37.732921]
21    [-122.087967, 37.670399]
22    [-122.265609, 37.797484]
23    [-122.267227, 37.828415]
24     [-122.38666, 37.599787]
25    [-122.401407, 37.789256]
26     [-122.283451, 37.87404]
27    [-122.024597, 38.003275]
28    [-122.183791, 37.878361]
29    [-122.056013, 37.928403]
30    [-121.945154, 38.018914]
31    [-122.299272, 37.903059]
32    [-

In [ ]:
gb_linac["coordinates"]

1     [-2.1352878, 57.1551578]
2     [-1.1108507, 51.2810411]
3           [-2.3910, 51.3917]
4       [-1.591675, 55.002836]
5      [-5.9416654, 54.587368]
                ...           
74     [-3.553979, 50.4817461]
75    [-5.0931924, 50.2668819]
76     [0.6883494, 51.5542562]
77      [-2.096532, 52.598801]
78    [-2.1807342, 52.1913636]
Name: coordinates, Length: 78, dtype: object

In [ ]:
linac_is_list=gb_linac["coordinates"].apply(lambda x: isinstance(x,list))
linac_is_str=gb_linac["coordinates"].apply(lambda x: isinstance(x,str))


is_list=df["coordinates"].apply(lambda x: isinstance(x,list))
is_str=df["coordinates"].apply(lambda x: isinstance(x,str))

print(f"df is list {is_list} and is string {is_str}")


linac is list 1     False
2     False
3     False
4     False
5     False
      ...  
74    False
75    False
76    False
77    False
78    False
Name: coordinates, Length: 78, dtype: bool and is string 1     True
2     True
3     True
4     True
5     True
      ... 
74    True
75    True
76    True
77    True
78    True
Name: coordinates, Length: 78, dtype: bool
df is list 0     True
1     True
2     True
3     True
4     True
5     True
6     True
7     True
8     True
9     True
10    True
11    True
12    True
13    True
14    True
15    True
16    True
17    True
18    True
19    True
20    True
21    True
22    True
23    True
24    True
25    True
26    True
27    True
28    True
29    True
30    True
31    True
32    True
33    True
34    True
35    True
36    True
37    True
38    True
39    True
40    True
41    True
42    True
43    True
Name: coordinates, dtype: bool and is string 0     False
1     False
2     False
3     False
4     False
5     False
6     False
7     Fal

In [ ]:
print(f"linac is list {linac_is_list} and is string {linac_is_str}\ndf is list {is_list} and is string {is_str}")

linac is list 1     False
2     False
3     False
4     False
5     False
      ...  
74    False
75    False
76    False
77    False
78    False
Name: coordinates, Length: 78, dtype: bool and is string 1     True
2     True
3     True
4     True
5     True
      ... 
74    True
75    True
76    True
77    True
78    True
Name: coordinates, Length: 78, dtype: bool
df is list 0     True
1     True
2     True
3     True
4     True
5     True
6     True
7     True
8     True
9     True
10    True
11    True
12    True
13    True
14    True
15    True
16    True
17    True
18    True
19    True
20    True
21    True
22    True
23    True
24    True
25    True
26    True
27    True
28    True
29    True
30    True
31    True
32    True
33    True
34    True
35    True
36    True
37    True
38    True
39    True
40    True
41    True
42    True
43    True
Name: coordinates, dtype: bool and is string 0     False
1     False
2     False
3     False
4     False
5     False
6     False
7     Fal

### OKAY SO IT WAS STRING REPRESENTATION VS AN ACTUAL LIST

## Now begins actual results again

so, now we take the gb linacs dataframe, and we can switch around the coordinates, then add a reversed `[long,lat]` list

In [ ]:
# Processing the gb linacs excel file so that we can get a dataframe that can be inputted into pydeck as a text layer, which im not sure O even like

gb_linac_prime = pd.read_excel("gbr_DIRAC.xlsx").drop(labels=0, axis=0)

gb_linac_text_layer = gb_linac_prime.copy(deep=True)
gb_linac_text_layer[["latitude","longitude"]] = gb_linac_text_layer["Coordinates"].str.split(',', expand=True)
gb_linac_text_layer["Coordinates"]=gb_linac_text_layer["Coordinates"].str.split(',').apply(
    lambda x: [float(x[1]), float(x[0])]
)

gb_linac_text_layer_latitude_order = gb_linac_text_layer.sort_values(by=["latitude"], ascending=False, ignore_index=True)

print(gb_linac_text_layer_latitude_order)                                                                       

gb_linac_text_layer = gb_linac_text_layer[["Operator Name","Coordinates"]].rename(columns={
    "Operator Name": "name",
    "Coordinates":"coordinates"}
    )


gb_linac_text_layer_latitude_order = gb_linac_text_layer_latitude_order[["Operator Name","Coordinates"]].rename(columns={
    "Operator Name": "name",
    "Coordinates":"coordinates"}
    )

gb_linac_text_layer_latitude_order["name"] = (gb_linac_text_layer_latitude_order.index + 1).astype(str)



               City                                      Operator Name  \
0         Inverness                   NHS Highland (Raigmore Hospital)   
1          Aberdeen           NHS Grampian  (Aberdeen Royal Infirmary)   
2            Dundee  NHS Tayside (Ninewells Hospital and Medical Sc...   
3         Edinburgh              NHS Lothian (Edinburgh Cancer Centre)   
4           Glasgow              NHS Glasgow (Beatson Oncology Center)   
..              ...                                                ...   
73           Exeter  Royal Devon And Exeter NHS Foundation Trust (E...   
74            Poole   Dorset Hospitals University NHS Foundation Trust   
75          Torquay  Torbay & South Devon Healthcare NHS Foundation...   
76         Plymouth  Plymouth Hospitals NHS Trust (Derriford Hospital)   
77  Truro, Cornwall                 Royal Cornwall Hospitals NHS Trust   

   He Photon And Electron Beam Rt Proton Ion Therapy XRay Generator  \
0                               2       

In [111]:
gb_linac_prime

,City,Operator Name,He Photon And Electron Beam Rt,Proton Ion Therapy,XRay Generator,Brachy Therapy Inc El,Last Update,Coordinates
1,Aberdeen,NHS Grampian (Aberdeen Royal Infirmary),3,0,0,1,2018,"57.1551578,-2.1352878"
2,Basingstoke,University Hospitals Southampton NHS Foundatio...,1,0,0,0,2023,"51.2810411,-1.1108507"
3,Bath,Royal United Hospital Bath NHS Trust,2,0,1,0,2018,"51.3917, -2.3910"
4,Bedlington,"Rutherford Health Proton Beam Therapy, Northum...",0,1,0,0,2020,"55.002836, -1.591675"
5,Belfast,Belfast HSC Trust (Belfast City Hospital),10,0,0,1,2019,"54.587368,-5.9416654"
...,...,...,...,...,...,...,...,...
74,Torquay,Torbay & South Devon Healthcare NHS Foundation...,2,0,0,0,2018,"50.4817461,-3.553979"
75,"Truro, Cornwall",Royal Cornwall Hospitals NHS Trust,2,0,1,0,2023,"50.2668819,-5.0931924"
76,Westcliff-on-Sea,Mid and South Essex NHS Foundation Trust (Sout...,4,0,0,1,2023,"51.5542562,0.6883494"
77,Wolverhampton,Royal Wolverhampton Hospitals NHS Trust,4,0,1,2,2016,"52.598801, -2.096532"


In [112]:
gb_linac_text_layer

,name,coordinates
1,NHS Grampian (Aberdeen Royal Infirmary),"[-2.1352878, 57.1551578]"
2,University Hospitals Southampton NHS Foundatio...,"[-1.1108507, 51.2810411]"
3,Royal United Hospital Bath NHS Trust,"[-2.391, 51.3917]"
4,"Rutherford Health Proton Beam Therapy, Northum...","[-1.591675, 55.002836]"
5,Belfast HSC Trust (Belfast City Hospital),"[-5.9416654, 54.587368]"
...,...,...
74,Torbay & South Devon Healthcare NHS Foundation...,"[-3.553979, 50.4817461]"
75,Royal Cornwall Hospitals NHS Trust,"[-5.0931924, 50.2668819]"
76,Mid and South Essex NHS Foundation Trust (Sout...,"[0.6883494, 51.5542562]"
77,Royal Wolverhampton Hospitals NHS Trust,"[-2.096532, 52.598801]"


In [113]:
gb_linac_text_layer_latitude_order

,name,coordinates
0,1,"[-4.1935703, 57.4747958]"
1,2,"[-2.1352878, 57.1551578]"
2,3,"[-3.0380032, 56.4635966]"
3,4,"[-3.2327131, 55.9617734]"
4,5,"[-4.313002, 55.88292]"
...,...,...
73,74,"[-3.508631, 50.717876]"
74,75,"[-2.44824, 50.712921]"
75,76,"[-3.553979, 50.4817461]"
76,77,"[-4.116519, 50.4171661]"


In [114]:
gb_linac_text_layer_latitude_order

,name,coordinates
0,1,"[-4.1935703, 57.4747958]"
1,2,"[-2.1352878, 57.1551578]"
2,3,"[-3.0380032, 56.4635966]"
3,4,"[-3.2327131, 55.9617734]"
4,5,"[-4.313002, 55.88292]"
...,...,...
73,74,"[-3.508631, 50.717876]"
74,75,"[-2.44824, 50.712921]"
75,76,"[-3.553979, 50.4817461]"
76,77,"[-4.116519, 50.4171661]"


In [121]:
from pydeck.types import String

MIN_POP, MAX_POP = 1, 50000.0  # pick what makes sense for your data
lp = np.log1p(df["population/km^2"].clip(lower=MIN_POP, upper=MAX_POP))
lp_min, lp_max = np.log1p(MIN_POP), np.log1p(MAX_POP)

z = (lp - lp_min) / (lp_max - lp_min)               # in [0, 1]

df_temp = df.copy(deep=True)

df_temp["alpha"] = (30 + z * (255 - 30)).round().astype(int)


h3_layer = pdk.Layer(
        "H3HexagonLayer",
        df_temp,
        pickable=True,
        stroked=False,
        filled=True,
        extruded=False,
        get_elevation="population/km^2",
        elevation_scale=20,
        high_precision=True,
        get_hexagon="h3",
        auto_highlight=True,
        get_fill_color="[0, 122, 255, alpha]",  # data-driven transparency
        get_line_color=[255, 255, 255],
        line_width_min_pixels=1,
    )

linacs = pdk.Layer(
        "TextLayer",
        gb_linac_text_layer_latitude_order,
        get_position="coordinates",
        get_text="name",
        get_size=10,
        get_color=[0, 0, 0],
        get_angle=0,
        get_text_anchor=String("start"),
        get_alignment_baseline=String("bottom"),
        size_min_pixels=0,
        size_max_pixels=10,
        background=True,
        get_background_color=[255,255,255,80]
    )

view_state = pdk.ViewState(latitude=54.4, longitude=-3.5, zoom=7, bearing=0, pitch=30)

r = pdk.Deck(layers=[h3_layer, linacs], 
             initial_view_state=view_state,
             map_provider="carto",
             map_style="light",  # road, dark, satellite, dark_no_labels, light_no_labels
)

r.to_html(filename="H3_maps/log_opacity_england_test.html", open_browser=True, notebook_display=False)


Dont really like the labels

Now adding in a hex layer of where the linacs are at!

In [133]:
def make_linac_h3_layer(prime_df, req_res):
    df_temp = prime_df.copy(deep=True)
    df_temp[["latitude","longitude"]] = df_temp["Coordinates"].str.split(',', expand=True).astype(float)
    df_temp["h3"] = df_temp.apply(lambda x: h3.latlng_to_cell(x.latitude,x.longitude, req_res), axis=1)
    df_temp = df_temp.drop(["Coordinates","latitude","longitude"], axis=1)
    return df_temp


In [134]:
gb_linac_h3_df = make_linac_h3_layer(gb_linac_prime,8)

In [135]:
gb_linac_h3_df

,City,Operator Name,He Photon And Electron Beam Rt,Proton Ion Therapy,XRay Generator,Brachy Therapy Inc El,Last Update,h3
1,Aberdeen,NHS Grampian (Aberdeen Royal Infirmary),3,0,0,1,2018,8819768cd5fffff
2,Basingstoke,University Hospitals Southampton NHS Foundatio...,1,0,0,0,2023,8819599115fffff
3,Bath,Royal United Hospital Bath NHS Trust,2,0,1,0,2018,881958232dfffff
4,Bedlington,"Rutherford Health Proton Beam Therapy, Northum...",0,1,0,0,2020,8819733505fffff
5,Belfast,Belfast HSC Trust (Belfast City Hospital),10,0,0,1,2019,881952d6adfffff
...,...,...,...,...,...,...,...,...
74,Torquay,Torbay & South Devon Healthcare NHS Foundation...,2,0,0,0,2018,88195b616bfffff
75,"Truro, Cornwall",Royal Cornwall Hospitals NHS Trust,2,0,1,0,2023,88187461d7fffff
76,Westcliff-on-Sea,Mid and South Essex NHS Foundation Trust (Sout...,4,0,0,1,2023,88194e2a4dfffff
77,Wolverhampton,Royal Wolverhampton Hospitals NHS Trust,4,0,1,2,2016,88195c0f49fffff


In [141]:
from pydeck.types import String

MIN_POP, MAX_POP = 1, 50000.0  # pick what makes sense for your data
lp = np.log1p(df["population/km^2"].clip(lower=MIN_POP, upper=MAX_POP))
lp_min, lp_max = np.log1p(MIN_POP), np.log1p(MAX_POP)

z = (lp - lp_min) / (lp_max - lp_min)               # in [0, 1]

df_temp = df.copy(deep=True)

df_temp["alpha"] = (30 + z * (255 - 30)).round().astype(int)


h3_layer = pdk.Layer(
        "H3HexagonLayer",
        df_temp,
        pickable=True,
        stroked=False,
        filled=True,
        extruded=False,
        get_elevation="population/km^2",
        elevation_scale=20,
        high_precision=True,
        get_hexagon="h3",
        auto_highlight=True,
        get_fill_color="[0, 122, 255, alpha]",  # data-driven transparency
        get_line_color=[255, 255, 255],
        line_width_min_pixels=1,
    )

linac_h3_layer = pdk.Layer(
        "H3HexagonLayer",
        gb_linac_h3_df,
        pickable=True,
        stroked=True,
        filled=True,
        extruded=False,
        #get_elevation="population/km^2",
        #elevation_scale=20,
        high_precision=True,
        get_hexagon="h3",
        auto_highlight=True,
        get_fill_color="[255, 0, 0]",  # data-driven transparency
        get_line_color=[255, 255, 255],
        line_width_min_pixels=1,
    )

linacs = pdk.Layer(
        "TextLayer",
        gb_linac_text_layer_latitude_order,
        get_position="coordinates",
        get_text="name",
        get_size=10,
        get_color=[0, 0, 0],
        get_angle=0,
        get_text_anchor=String("start"),
        get_alignment_baseline=String("bottom"),
        size_min_pixels=10,
        size_max_pixels=16,
        background=True,
        get_background_color=[255,255,255,180]
    )



view_state = pdk.ViewState(latitude=54.4, longitude=-3.5, zoom=7, bearing=0, pitch=30)

r = pdk.Deck(layers=[h3_layer, linac_h3_layer, linacs], 
             initial_view_state=view_state,
             map_provider="carto",
             map_style="light",  # road, dark, satellite, dark_no_labels, light_no_labels
)

r.to_html(filename="H3_maps/log_opacity_england_test.html", open_browser=True, notebook_display=False)
